<a href="https://colab.research.google.com/github/aljeralmazan28-web/Algoverse-Research-TNAM/blob/main/Baseline_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MOLI_ZIP = '/content/drive/MyDrive/MOLI.zip'        # edit
SEX_CSV  = None                                     # set later to your sex.csv path
DRUG     = 'Paclitaxel'
DATA_DIR = '/content/moli_data'

!unzip -q -o "{MOLI_ZIP}" -d /content
import sys; sys.path.insert(0, '/content/MOLI')     # gives us utils.py

# download only what the expression-only baseline needs (~2 GB, a few minutes)
!mkdir -p {DATA_DIR}
!wget -q -O {DATA_DIR}/exprs_homogenized.tar.gz "https://zenodo.org/records/4036592/files/exprs_homogenized.tar.gz?download=1"
!wget -q -O {DATA_DIR}/response.tar.gz "https://zenodo.org/records/4036592/files/response.tar.gz?download=1"
!tar -xzf {DATA_DIR}/exprs_homogenized.tar.gz -C {DATA_DIR}
!tar -xzf {DATA_DIR}/response.tar.gz -C {DATA_DIR}
!find {DATA_DIR} -name "*{DRUG}*" | head -30

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
^C
^C
/content/moli_data/response/GSE25065_response.Paclitaxel.tsv
/content/moli_data/response/GDSC_response.Paclitaxel.tsv
/content/moli_data/response/PDX_response.Paclitaxel.tsv
/content/moli_data/response/TCGA_response.Paclitaxel.tsv
/content/moli_data/response/PDX_response.multi-OMICS.Paclitaxel.tsv
/content/moli_data/response/CCLE_response.Paclitaxel.tsv
/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.TCGA_exprs.Paclitaxel.tsv


In [5]:
import random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from utils import AllTripletSelector

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [7]:
from glob import glob

# search recursively, since the tarballs unpack into subfolders; prefer the PDX-harmonized file
e_files = [f for f in glob(f"{DATA_DIR}/**/GDSC_exprs.{DRUG}.eb_with.*", recursive=True) if '.z.' not in f]
e_files = sorted(e_files, key=lambda f: 'PDX' not in f)
r_files = glob(f"{DATA_DIR}/**/GDSC_response.{DRUG}.tsv", recursive=True)
print("expression files:", e_files); print("response files:", r_files)
assert e_files and r_files, "Files not found: check the !find output above for the real names."

E = pd.read_csv(e_files[0], sep="\t", index_col=0, decimal=",").T   # rows = cell lines, cols = genes
R = pd.read_csv(r_files[0], sep="\t", index_col=0, decimal=",")
E.index = E.index.astype(str); R.index = R.index.astype(str)        # IDs can load as str vs int

resp = R['response'] if 'response' in R.columns else R.iloc[:, 0]
y_all = resp.map({'R': 0, 'S': 1}).dropna().astype(int)
ids = E.index.intersection(y_all.index)
E, y_all = E.loc[ids], y_all.loc[ids]
print(E.shape, "| sensitive:", y_all.sum(), "| resistant:", (y_all == 0).sum())

E = E.dropna(axis=1)   # drop genes with missing values

expression files: ['/content/moli_data/exprs_homogenized/GDSC_exprs.Paclitaxel.eb_with.TCGA_exprs.Paclitaxel.tsv']
response files: ['/content/moli_data/response/GDSC_response.Paclitaxel.tsv']
(389, 11306) | sensitive: 26 | resistant: 363


In [8]:
class Encoder(nn.Module):
    def __init__(self, in_dim, h_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, h_dim), nn.BatchNorm1d(h_dim),
                                 nn.ReLU(), nn.Dropout())
    def forward(self, x): return self.net(x)

class Classifier(nn.Module):
    def __init__(self, h_dim, rate):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(h_dim, 1), nn.Dropout(rate), nn.Sigmoid())
    def forward(self, x): return self.net(x)

triplet_selector = AllTripletSelector()

def train_fold(Xtr, ytr, Xte, p, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)

    # filter + scale fit on the training fold only
    vt = VarianceThreshold(0.05).fit(Xtr)
    Xtr, Xte = vt.transform(Xtr), vt.transform(Xte)
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

    # class-balanced sampling
    counts = np.array([(ytr == c).sum() for c in np.unique(ytr)])
    w = (1.0 / counts)[ytr]
    sampler = WeightedRandomSampler(torch.from_numpy(w).double(), len(w), replacement=True)
    loader = DataLoader(TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                                      torch.tensor(ytr, dtype=torch.float32)),
                        batch_size=p['mb_size'], sampler=sampler, drop_last=True)

    enc, clf = Encoder(Xtr.shape[1], p['h_dim']), Classifier(p['h_dim'], p['rate'])
    opt_e = optim.Adagrad(enc.parameters(), lr=p['lr_enc'])
    opt_c = optim.Adagrad(clf.parameters(), lr=p['lr_clf'], weight_decay=p['wd'])
    trip = nn.TripletMarginLoss(margin=p['marg'], p=2)
    bce = nn.BCELoss()

    for epoch in range(p['epochs']):
        enc.train(); clf.train()
        for xb, yb in loader:
            if yb.mean() in (0., 1.):            # need both classes in the batch
                continue
            z = enc(xb); pred = clf(z)
            loss = bce(pred, yb.view(-1, 1))
            t = triplet_selector.get_triplets(z, yb)
            if t.dim() == 2 and len(t) > 0:
                loss = loss + p['lam'] * trip(z[t[:, 0]], z[t[:, 1]], z[t[:, 2]])
            opt_e.zero_grad(); opt_c.zero_grad()
            loss.backward()
            opt_e.step(); opt_c.step()

    enc.eval(); clf.eval()
    with torch.no_grad():
        return clf(enc(torch.tensor(Xte, dtype=torch.float32))).numpy().ravel()

In [9]:
def run_cv(p, X=E, y=y_all, n_splits=5, seed=SEED):
    Xv, yv = X.values.astype(np.float32), y.values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(yv)); aucs = []
    for k, (tr, te) in enumerate(skf.split(Xv, yv), 1):
        oof[te] = train_fold(Xv[tr], yv[tr], Xv[te], p, seed)
        aucs.append(roc_auc_score(yv[te], oof[te]))
        print(f"  fold {k}: AUROC={aucs[-1]:.3f}")
    return oof, aucs

params = dict(mb_size=36, h_dim=256, marg=1.0, lr_enc=0.01, lr_clf=0.01,
              epochs=50, rate=0.5, wd=0.001, lam=0.1)

# smoke test first: 3 epochs, confirms nothing crashes (AUROC will be rough)
_ = run_cv({**params, 'epochs': 3})

  fold 1: AUROC=0.521
  fold 2: AUROC=0.704
  fold 3: AUROC=0.726
  fold 4: AUROC=0.653
  fold 5: AUROC=0.822


In [10]:
# real run
oof, aucs = run_cv(params)
print(f"\nMean AUROC {np.mean(aucs):.3f} +- {np.std(aucs):.3f} | "
      f"pooled AUPRC {average_precision_score(y_all.values, oof):.3f}")

  fold 1: AUROC=0.614
  fold 2: AUROC=0.586
  fold 3: AUROC=0.770
  fold 4: AUROC=0.613
  fold 5: AUROC=0.828

Mean AUROC 0.682 +- 0.097 | pooled AUPRC 0.131


In [11]:
SEX_CSV = None   # e.g. '/content/drive/MyDrive/sex.csv' (cell line ID -> F/M)

sex = pd.read_csv(SEX_CSV, index_col=0).iloc[:, 0] if SEX_CSV else None
if sex is not None:
    sex.index = sex.index.astype(str)
    sx = sex.reindex(y_all.index).values
    yt = y_all.values

    def auroc(mask):
        return roc_auc_score(yt[mask], oof[mask]) if len(set(yt[mask])) == 2 else np.nan

    for g in ['F', 'M']:
        m = sx == g
        print(f"{g}: n={m.sum()}  AUROC={auroc(m):.3f}")

    rng = np.random.default_rng(0); gaps = []
    f_idx, m_idx = np.where(sx == 'F')[0], np.where(sx == 'M')[0]
    for _ in range(1000):
        fb, mb = rng.choice(f_idx, len(f_idx)), rng.choice(m_idx, len(m_idx))
        if len(set(yt[fb])) == 2 and len(set(yt[mb])) == 2:
            gaps.append(roc_auc_score(yt[fb], oof[fb]) - roc_auc_score(yt[mb], oof[mb]))
    print(f"F - M gap: {np.mean(gaps):.3f}  95% CI "
          f"[{np.percentile(gaps, 2.5):.3f}, {np.percentile(gaps, 97.5):.3f}]")
else:
    print("No sex file provided; skipping.")

No sex file provided; skipping.
